# 🔬 Breast Cancer (BUSI) Ablation Phase 2

This notebook sequentially trains **Experiment B** (MLP without Conv) and **Experiment D** (Spatially-Aware MoE) on Kaggle.
All the code is downloaded directly from the GitHub repository.

In [ ]:
# 1. Clone Repo and Install Dependencies
!rm -rf vision_tranformer_moe
!git clone https://github.com/toqeer-ahmed/vision_tranformer_moe.git
%cd vision_tranformer_moe
!pip install -r requirements.txt
!pip install kaggle albumentations tensorboard transformers torch torchvision

## 2. Authenticate with Kaggle and Download Dataset


In [ ]:
import os
import shutil
import glob

# Set the provided Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = "KGAT_f92b021c2b42601bd960c76192014a55"
!mkdir -p ~/.kaggle
!echo "KGAT_f92b021c2b42601bd960c76192014a55" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

# Download and extract dataset
!kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset
!unzip -q breast-ultrasound-images-dataset.zip -d busi_temp

dest_dir = "data/medical_dataset"
os.makedirs(os.path.join(dest_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(dest_dir, "masks"), exist_ok=True)

src_dir = "busi_temp/Dataset_BUSI_with_GT"
for category in ["benign", "malignant", "normal"]:
    cat_path = os.path.join(src_dir, category)
    if not os.path.exists(cat_path):
        continue
    all_files = glob.glob(os.path.join(cat_path, "*.png"))
    images = [f for f in all_files if "mask" not in f]
    for img_path in images:
        base = os.path.splitext(os.path.basename(img_path))[0]
        shutil.copy(img_path, os.path.join(dest_dir, "images", f"{category}_{base}.png"))
        mask_path = os.path.join(cat_path, f"{base}_mask.png")
        if os.path.exists(mask_path):
            shutil.copy(mask_path, os.path.join(dest_dir, "masks", f"{category}_{base}_mask.png"))

print("BUSI Dataset successfully formatted!")

## 3. Launch Training Run


In [ ]:
!PYTHONPATH=. python scripts/run_ablation_phase2.py

## 4. Zip and Export Results


In [ ]:
import shutil
import IPython
shutil.make_archive("/kaggle/working/ablation_phase2_results", "zip", "outputs/")
IPython.display.FileLink("/kaggle/working/ablation_phase2_results.zip")